# M04 Using Dictionary Learning Features as Classifiers


## Probe on top of features

Train a tiny classifier on synthetic feature activations and inspect which recovered directions matter most.


In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(31)

feature_names = ["calendar", "citation", "refusal", "tool"]
features = torch.randn(400, 4)
target = 1.4 * features[:, 2] + 0.8 * features[:, 3] - 0.9 * features[:, 1]
labels = (torch.sigmoid(target) > 0.5).float().unsqueeze(1)

probe = torch.nn.Linear(4, 1)
optimizer = torch.optim.Adam(probe.parameters(), lr=0.05)

for step in range(500):
    logits = probe(features)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    probs = torch.sigmoid(probe(features))
    accuracy = ((probs > 0.5).float() == labels).float().mean().item()
    weights = probe.weight.flatten()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(feature_names, weights.tolist(), color=["#4c78a8", "#9ecae9", "#f58518", "#54a24b"])
ax.axhline(0, color="0.75", linewidth=1)
ax.set_title("Probe weights over recovered features")
plt.tight_layout()

print("Final BCE loss:", float(loss.detach()))
print("Accuracy:", round(accuracy, 3))
print("Weights:", dict(zip(feature_names, [round(value, 3) for value in weights.tolist()])))


## Takeaway

Feature quality becomes measurable once features can support a readout task.


## Turn this notebook into research output

Research worksheet means this notebook is not complete when the cells finish. Use the templates in /templates and leave behind written outputs.


### Before you run

- Pick one baseline and one possible confounder before you fit the probe.
- Write down what high accuracy would and would not prove.
- Decide which feature weight you expect to dominate.


### After you run

- Explain why the strongest probe weight might still be misleading.
- List three confounders or alternative explanations for the result.


### Ship these artifacts

- One probe memo with baseline and confounders.
- One figure of feature weights.
- One proposed follow-up label or task.


## Self-check questions

- If features can feed a classifier directly, what does that show, and what does it still not show?
- If the probe accuracy is high, which confounders would you check before deciding the result is trustworthy?
- Besides accuracy, which baseline or comparison would you use to judge whether the features really have readout value?
- If you cannot answer at least two of these without rereading the note, revisit the article and your written outputs.
